# Chapter 11 — Agent Architecture & Agent Evaluation

*Where we are:* a **fixed** RAG pipeline always runs the same steps; an **agent** *chooses* tools
based on the query.

```
query → [ planner → tool calls (schema-validated) → trajectory ] → grounded answer
```

We build **one controlled, deterministic** agent — tool schemas, a tool registry, the planner, the
run loop with full trajectory capture, and the evaluation metrics — **inline** (packaged in
`patentrag.agent` / `patentrag.evaluation`).

In [1]:
# === Chapter 11 · standard bootstrap (identical pattern in every notebook) ===
# Runs standalone on a fresh Google Colab VM *or* a local checkout.
import os, sys, subprocess

REPO_URL = "https://github.com/rsalehin/patent-rag-masterclass"
NEED_OCR = False
IN_COLAB = "google.colab" in sys.modules


def _clone_repo(url, target):
    """Clone the repo on Colab. For a PRIVATE repo, authenticate with a GitHub token read from
    Colab Secrets (key 'GITHUB_TOKEN') or the GITHUB_TOKEN env var. The token is never printed."""
    token = None
    try:
        from google.colab import userdata  # type: ignore
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = os.environ.get("GITHUB_TOKEN")
    auth_url = url
    if token and url.startswith("https://github.com/"):
        auth_url = url.replace("https://github.com/", f"https://{token}@github.com/")
    r = subprocess.run(["git", "clone", "--depth", "1", auth_url, target],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)  # avoid leaking the token
    if r.returncode != 0:
        raise RuntimeError(
            "git clone failed. This is a PRIVATE repo, so Colab needs a GitHub token:\n"
            "  1) Create a token (scope: repo) at https://github.com/settings/tokens\n"
            "  2) In Colab, open the key icon (Secrets) in the left sidebar, add a secret named\n"
            "     GITHUB_TOKEN, paste the token, and enable 'Notebook access'.\n"
            "  3) Re-run this cell.\n"
            "  (Alternatively, make the GitHub repo public — then no token is needed.)")


if IN_COLAB:
    target = "/content/patent-rag-masterclass"
    if not os.path.isdir(target):
        if not REPO_URL:
            raise RuntimeError("Set REPO_URL to this repo's GitHub URL (see README.md).")
        _clone_repo(REPO_URL, target)
    os.chdir(target)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if NEED_OCR:
        subprocess.run(["apt-get", "install", "-y", "-q", "tesseract-ocr"], check=False)

# Ensure the repo root (containing patentrag/) is importable.
for _cand in [os.getcwd()] + [os.path.dirname(os.getcwd())]:
    if os.path.isdir(os.path.join(_cand, "patentrag")):
        if _cand not in sys.path:
            sys.path.insert(0, _cand)
        break

from patentrag import bootstrap as bs
bs.setup_environment(REPO_URL, need_ocr=NEED_OCR)
bs.set_seeds()
_env = bs.environment_report()
print("Chapter 11 bootstrap OK")
print("  Python", _env["python"], "| Colab:", _env["in_colab"], "| CPU cores:", _env["cpu_count"])
print("  torch", _env["torch"], "| CUDA:", _env["cuda_available"], "| tesseract:", _env["tesseract"])

Chapter 11 bootstrap OK
  Python 3.12.10 | Colab: False | CPU cores: 24
  torch 2.12.0.dev20260304+cu130 | CUDA: True | tesseract: True


## 55. Tool schemas (Pydantic / JSON Schema)

Every tool has a strict **Pydantic argument schema** — malformed calls are rejected before
execution. A field validator enforces patent-number syntax.

In [2]:
import re
from pydantic import BaseModel, Field, field_validator

class SearchPatentsArgs(BaseModel):                      # arguments for a "search_patents" tool
    query: str = Field(min_length=2, max_length=400)     # bounded query string
    top_k: int = Field(default=5, ge=1, le=50)           # how many results (1..50)

class FetchPatentArgs(BaseModel):                        # arguments for a "fetch_patent" tool
    publication_number: str                              # e.g. "US9081550B2"
    @field_validator("publication_number")               # custom validation of the format
    @classmethod
    def _valid(cls, v):
        if not re.match(r"^[A-Z]{2}\d{5,}[A-Z]\d?$", v.strip()):   # 2 letters + digits + kind code
            raise ValueError(f"invalid patent publication number: {v!r}")
        return v.strip()

import json
print("search_patents JSON schema props:", json.dumps(SearchPatentsArgs.model_json_schema()["properties"]))
try:
    FetchPatentArgs(publication_number="not-a-number")   # should be rejected by the validator
except Exception as e:
    print("rejected bad fetch_patent arg ->", type(e).__name__)

search_patents JSON schema props: {"query": {"maxLength": 400, "minLength": 2, "title": "Query", "type": "string"}, "top_k": {"default": 5, "maximum": 50, "minimum": 1, "title": "Top K", "type": "integer"}}
rejected bad fetch_patent arg -> ValidationError


## Tools + a registry

A **tool** bundles a name, its argument schema, the function, and a **required permission**
(consumed by Chapter 12's guardrails). The **registry** validates arguments before dispatching.
Inline == `patentrag.agent.Tool` / `ToolRegistry`.

In [3]:
from dataclasses import dataclass
from typing import Callable, Any
from patentrag.dense import DenseRetriever
from patentrag.fusion import reciprocal_rank_fusion

# --- wire up retrieval so the tools have something to search ---
docs = {d.doc_id: d for d in bs.ensure("docs_canonical")}     # doc_id -> PatentDocument
chunks = bs.ensure("chunks"); by_id = {c.chunk_id: c for c in chunks}
bm25 = bs.ensure("bm25_index"); emb = bs.ensure("embeddings")
dense = DenseRetriever(emb["chunk_ids"], emb["matrix"])
def hybrid(query, top_k):                                     # fuse BM25 + dense, return top_k chunks
    b = [c for c, _ in bm25.search(query, 40)]; d = [c for c, _ in dense.search(query, 40)]
    return [by_id[cid] for cid, _ in reciprocal_rank_fusion([b, d])[:top_k]]

@dataclass
class Tool:                                                   # a callable tool with a schema + permission
    name: str
    schema: type
    fn: Callable[..., Any]
    required_permission: str = "read"

class ToolRegistry:                                          # holds tools; validates args before calling
    def __init__(self): self._tools = {}
    def register(self, tool): self._tools[tool.name] = tool
    def get(self, name): return self._tools[name]
    def names(self): return sorted(self._tools)
    def call(self, name, args):
        tool = self._tools[name]                             # look up the tool
        validated = tool.schema(**args)                      # validate/coerce args (raises on bad input)
        return tool.fn(**validated.model_dump())             # dispatch with clean kwargs

# --- define the tools (read-only patent operations) ---
def search_patents(query, top_k=5):                          # return distinct patents matching a query
    seen, out = set(), []
    for c in hybrid(query, top_k * 4):
        if c.document_id in seen: continue
        seen.add(c.document_id); out.append({"publication_number": docs[c.document_id].publication_number,
                                             "title": docs[c.document_id].title})
        if len(out) >= top_k: break
    return out
def fetch_patent(publication_number):                        # look up one patent's biblio
    d = next((x for x in docs.values() if x.publication_number == publication_number), None)
    return {"error": "not found"} if not d else {"publication_number": d.publication_number,
            "title": d.title, "abstract": d.abstract[:120], "n_claims": len(d.claims)}
def retrieve_passages(query, top_k=8):                       # grounding passages for the final answer
    return [{"chunk_id": c.chunk_id, "publication_number": c.publication_number, "text": c.text} for c in hybrid(query, top_k)]

class RetrievePassagesArgs(BaseModel):
    query: str = Field(min_length=2, max_length=400); top_k: int = Field(default=8, ge=1, le=50)

registry = ToolRegistry()
registry.register(Tool("search_patents", SearchPatentsArgs, search_patents))
registry.register(Tool("fetch_patent", FetchPatentArgs, fetch_patent))
registry.register(Tool("retrieve_passages", RetrievePassagesArgs, retrieve_passages))
print("tools:", registry.names())
print("call fetch_patent:", registry.call("fetch_patent", {"publication_number": "US9081550B2"})["title"])

tools: ['fetch_patent', 'retrieve_passages', 'search_patents']
call fetch_patent: Adding speech capabilities to existing computer applications with complex graphical user interfaces


## 56. The agent — planner, run loop, trajectory

The planner maps **query features** to an ordered tool plan (a named patent → `fetch_patent`; the
word "claim" → a claims search; always end by retrieving grounding passages). Each step records
`(tool, args, result, error)` — the full **trajectory**. Inline == `patentrag.agent.PatentAgent`.

In [4]:
_PUBNUM = re.compile(r"\b([A-Z]{2}\d{5,}[A-Z]\d?)\b")        # detects a patent number inside the query

@dataclass
class AgentStep:                                             # one recorded tool invocation
    tool: str; args: dict; result: Any = None; error: str = None

class PatentAgent:
    def __init__(self, registry, generate_fn=None):
        self.registry = registry; self.generate_fn = generate_fn   # registry + optional answer generator
    def plan(self, query):                                  # choose an ordered list of (tool, args)
        plan = []
        m = _PUBNUM.search(query)
        if m:                                               # a specific patent is named -> fetch it first
            plan.append(("fetch_patent", {"publication_number": m.group(1)}))
        if re.search(r"\bclaim", query, re.I):              # asks about claims -> search claims
            plan.append(("search_patents", {"query": query, "top_k": 5}))
        plan.append(("retrieve_passages", {"query": query, "top_k": 8}))   # always ground the answer
        return plan
    def run(self, query):                                   # execute the plan, capturing every step
        steps, passages = [], []
        for tool, args in self.plan(query):
            step = AgentStep(tool=tool, args=args)
            try:
                step.result = self.registry.call(tool, args)   # validated dispatch
                if tool == "retrieve_passages": passages = step.result
            except Exception as e:                          # capture failures into the trajectory (don't crash)
                step.error = f"{type(e).__name__}: {e}"
            steps.append(step)
        final = self.generate_fn(query, passages) if (self.generate_fn and passages) else None
        return {"query": query, "steps": steps, "tool_sequence": [s.tool for s in steps], "final": final}

from patentrag.generation import generate_answer, MockLLMProvider
def gen(q, passages):                                       # turn retrieved passages into a grounded answer
    return generate_answer(q, [by_id[p["chunk_id"]] for p in passages], provider=MockLLMProvider())

agent = PatentAgent(registry, generate_fn=gen)
traj = agent.run("What does patent US9081550B2 disclose in its claims about voice interfaces?")
print("tool sequence:", traj["tool_sequence"])
for s in traj["steps"]:                                     # print each step's tool + a snippet of its result
    print(f"  → {s.tool:18} {str(s.result)[:64] if s.result else s.error}")
print("final answer citations:", len(traj["final"].citations))

C:\Users\rsalehin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8176.03it/s]

tool sequence: ['fetch_patent', 'search_patents', 'retrieve_passages']
  → fetch_patent       {'publication_number': 'US9081550B2', 'title': 'Adding speech ca
  → search_patents     [{'publication_number': 'US9081550B2', 'title': 'Adding speech c
  → retrieve_passages  [{'chunk_id': '20cb637155af', 'publication_number': 'US9081550B2
final answer citations: 2


## 57–62. Agent evaluation

Agent quality is **more than the final answer** — evaluate tool *selection*, *arguments*, and the
*trajectory*. We inline the metrics (packaged in `patentrag.evaluation`).

In [5]:
from collections import Counter
import pandas as pd

def tool_prf(predicted, gold):                              # precision/recall/F1 over tool-name multisets
    p, g = Counter(predicted), Counter(gold)
    tp = sum((p & g).values())                              # matched tool calls
    prec = tp / max(1, sum(p.values())); rec = tp / max(1, sum(g.values()))
    f1 = 0.0 if prec + rec == 0 else 2 * prec * rec / (prec + rec)
    return {"precision": prec, "recall": rec, "f1": f1}

def trajectory_match(predicted, gold, mode="exact"):        # compare tool sequences
    if mode == "exact":     return predicted == gold        # same tools, same order
    if mode == "unordered": return Counter(predicted) == Counter(gold)   # same multiset
    if mode == "subset":    return set(gold).issubset(set(predicted))    # gold tools all present
    raise ValueError(mode)

gold = [                                                    # (query, expected tool sequence)
    ("Summarize what patent US9081550B2 covers.", ["fetch_patent", "retrieve_passages"]),
    ("Which claims mention nearest neighbor search?", ["search_patents", "retrieve_passages"]),
    ("Explain retrieval-aware embeddings.", ["retrieve_passages"]),
]
rows = []
for q, gtools in gold:
    pred = agent.run(q)["tool_sequence"]                    # what the agent actually did
    rows.append({"query": q[:34], "gold": " → ".join(gtools), "predicted": " → ".join(pred),
                 "tool_F1": round(tool_prf(pred, gtools)["f1"], 2),
                 "exact": trajectory_match(pred, gtools, "exact"),
                 "subset": trajectory_match(pred, gtools, "subset")})
agent_eval = pd.DataFrame(rows); agent_eval

,query,gold,predicted,tool_F1,exact,subset
0,Summarize what patent US9081550B2,fetch_patent → retrieve_passages,fetch_patent → retrieve_passages,1.0,True,True
1,Which claims mention nearest neigh,search_patents → retrieve_passages,search_patents → retrieve_passages,1.0,True,True
2,Explain retrieval-aware embeddings,retrieve_passages,retrieve_passages,1.0,True,True


In [6]:
import numpy as np
# argument correctness: did fetch_patent get the exact number the query named?
t = agent.run("What is in patent US11971885B2?")
fetch_args = [s.args for s in t["steps"] if s.tool == "fetch_patent"]
arg_ok = float(fetch_args == [{"publication_number": "US11971885B2"}])
print("argument correctness:", arg_ok)
print("mean tool-call F1:", round(np.mean([r["tool_F1"] for r in rows]), 2))
print("avg steps:", round(np.mean([len(agent.run(q)["steps"]) for q, _ in gold]), 1))

argument correctness: 1.0
mean tool-call F1: 1.0
avg steps: 1.7


> Inline `Tool`/`ToolRegistry`, the tools, `PatentAgent` and the metrics above ==
> `patentrag.agent` + `patentrag.evaluation`.

**Production implications.** Constrain tools by permission (Ch 12), validate every argument, cap
steps to prevent loops, and log the full trajectory for offline eval. Tool-call F1 and argument
correctness are the leading indicators; final-answer quality is the lagging one.

## Chapter invariants

In [7]:
assert set(registry.names()) >= {"fetch_patent", "search_patents", "retrieve_passages"}
assert traj["tool_sequence"][0] == "fetch_patent"           # pub-number query -> fetch first
assert "retrieve_passages" in traj["tool_sequence"]          # always grounds
assert traj["final"] is not None and traj["final"].citations # produced a grounded answer
assert arg_ok == 1.0                                         # right patent number extracted
assert trajectory_match(["a", "b"], ["b", "a"], "unordered") and not trajectory_match(["a"], ["a", "b"], "exact")
from patentrag.agent import PatentAgent as PkgAgent          # packaged version exists
print("All Chapter 11 invariants hold.")

All Chapter 11 invariants hold.


In [8]:
# === Chapter 11 validation footer ===
import time, platform, sys, importlib.metadata as _md
_pkgs = ['pydantic', 'pandas', 'numpy']
print("Chapter 11 — environment")
print("  Python :", sys.version.split()[0], "on", platform.system(), platform.release())
for _p in _pkgs:
    try: print(f"  {_p:24}: {_md.version(_p)}")
    except Exception: print(f"  {_p:24}: (not installed)")
print()
print("CHAPTER 11 VALIDATION: PASS")

Chapter 11 — environment
  Python : 3.12.10 on Windows 11
  pydantic                : 2.13.3
  pandas                  : 3.0.2
  numpy                   : 2.4.2

CHAPTER 11 VALIDATION: PASS
